# Build 1 — Lakebase execution evidence

Executed against Lakebase project **volta-plant-floor** (branch `production`, db `databricks_postgres`) at 2026-08-31T00:26:44.426377+00:00.

Proves execution of: **(A)** the operational schema (related domain tables + primary keys + joins);
**(B)** the **branch's** `next_shift_forecast` migration/forecast (developed on the Lakebase `dev`
branch, promoted to `production`); **(C)** a separate **writable** table (`ops.work_orders`) distinct
from the **read-only, sync-managed** mirrors. Outputs are the real returned results.

In [1]:
import os, psycopg
conn = psycopg.connect(host=os.environ['PGHOST'], user=os.environ['PGUSER'],
                       password=os.environ['PGPASSWORD'], dbname='databricks_postgres', sslmode='require')
cur = conn.cursor(); cur.execute('SELECT version()'); print(cur.fetchone()[0])

PostgreSQL 17.11 (32e7196) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## A. Operational schema — related tables and their primary keys

In [2]:
cur.execute(SCHEMA_KEYS_SQL); import json; print(json.dumps(cur.fetchall(), indent=2))

[
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "line_status",
    "pk_constraint": "line_status_pkey",
    "key_columns": "line_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "maintenance_recommendations",
    "pk_constraint": "maintenance_recommendations_pkey",
    "key_columns": "line_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "open_atrisk",
    "pk_constraint": "open_atrisk_pkey",
    "key_columns": "line_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "parts",
    "pk_constraint": "parts_pkey",
    "key_columns": "part_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "parts_search",
    "pk_constraint": "parts_search_pkey",
    "key_columns": "part_id"
  },
  {
    "schema": "ops",
    "table": "work_orders",
    "pk_constraint": "work_orders_pkey",
    "key_columns": "wo_id"
  }
]


Relationships via keys: `line_status`/`open_atrisk`/`maintenance_recommendations` share **line_id**;
`open_atrisk.candidate_part_id` → `parts.part_id`. Join across the domain (hero LINE-0004):

In [3]:
cur.execute(RELATIONAL_JOIN_SQL); print(cur.fetchall())

[
  {
    "line_id": "LINE-0004",
    "plant_id": "PLANT-03",
    "risk_band": "critical",
    "part_local": false,
    "candidate_part_id": "PART-00001",
    "part_name": "hydraulic_seal Injection_Molder",
    "lead_time_days": 2,
    "recommended_action": "pull_now",
    "predicted_downtime_cost_avoided_usd": 76560.0
  }
]


## B. Branch change executed — `next_shift_forecast` (migration + forecast)

Developed on the Lakebase **`dev`** branch (copy-on-write off `production`), validated, then promoted
to `production` (migration `agent_change/001_create_next_shift_forecast.sql`). It projects next-shift
(8h) unplanned-downtime cost by plant from the synced `line_status`. Confirm the object exists and query it:

In [4]:
cur.execute("SELECT table_schema, table_name FROM information_schema.tables"
            " WHERE table_name='next_shift_forecast'"); print('DDL applied:', cur.fetchall())
cur.execute('SELECT * FROM %s.next_shift_forecast ORDER BY projected_next_shift_cost_usd DESC' % SCHEMA)
print(cur.fetchall())

DDL applied: [
  {
    "table_schema": "dev_manffred_calvosanchez_volta_industrial",
    "table_name": "next_shift_forecast"
  }
]
[
  {
    "plant_id": "PLANT-01",
    "atrisk_lines": 14,
    "projected_next_shift_cost_usd": "2071872"
  },
  {
    "plant_id": "PLANT-06",
    "atrisk_lines": 14,
    "projected_next_shift_cost_usd": "2058320"
  },
  {
    "plant_id": "PLANT-03",
    "atrisk_lines": 13,
    "projected_next_shift_cost_usd": "1973136"
  },
  {
    "plant_id": "PLANT-05",
    "atrisk_lines": 13,
    "projected_next_shift_cost_usd": "1904320"
  },
  {
    "plant_id": "PLANT-07",
    "atrisk_lines": 12,
    "projected_next_shift_cost_usd": "1780416"
  },
  {
    "plant_id": "PLANT-04",
    "atrisk_lines": 11,
    "projected_next_shift_cost_usd": "1685728"
  },
  {
    "plant_id": "PLANT-02",
    "atrisk_lines": 9,
    "projected_next_shift_cost_usd": "1420144"
  },
  {
    "plant_id": "PLANT-08",
    "atrisk_lines": 8,
    "projected_next_shift_cost_usd": "1188704"
  }
]


## C. Separate WRITABLE table (`ops.work_orders`) distinct from read-only synced mirrors

### C1. Writes succeed on the operational table

In [5]:
cur.execute("INSERT INTO ops.work_orders (line_id, action_type, status)"
            " VALUES ('LINE-TEST','pull_now','proposed') RETURNING wo_id, line_id, action_type, status, created_at")
print('INSERT ok:', cur.fetchone())
cur.execute("DELETE FROM ops.work_orders WHERE line_id='LINE-TEST' RETURNING wo_id"); print('DELETE ok:', cur.fetchone())

INSERT ok: [
  {
    "wo_id": 6,
    "line_id": "LINE-TEST",
    "action_type": "pull_now",
    "status": "proposed",
    "created_at": "2026-08-31 00:23:09.971729+00:00"
  }
]
DELETE ok: [
  {
    "wo_id": 6
  }
]


### C2. The synced tables are read-only, sync-managed mirrors (managed status of `line_status`)

In [6]:
print(get_synced_table_status('line_status'))

{
  "detailed_state": "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE",
  "message": "Online Table creation succeeded using Delta Live Tables: https://fevm-serverless-stable-casaman.cloud.databricks.com#joblist/pipelines/4e987a21-4778-4b07-90c6-3c4eef60662b/updates/78b8c792-fb74-4b22-90d6-bd5ea9d74701.",
  "data_synchronization_status": true
}


### C3. Classification — writable operational tables vs read-only synced mirrors

In [7]:
import json; print(json.dumps(classification, indent=2))

{
  "read_only_synced_mirrors (managed by postgres_synced_tables; source of truth in UC gold)": [
    "line_status",
    "open_atrisk",
    "maintenance_recommendations",
    "parts"
  ],
  "writable_operational_tables (app/agent-owned, not synced)": [
    "ops.work_orders",
    "dev_manffred_calvosanchez_volta_industrial.parts_search",
    "dev_manffred_calvosanchez_volta_industrial.next_shift_forecast"
  ],
  "note": "Synced tables are read-only by contract: managed by the sync pipeline (mirrors of UC gold). Direct Postgres writes are unsupported and overwritten on the next sync. ops.work_orders has no sync and is the writable operational table (also the Lakebase CDF source)."
}


## Summary

- 6 domain tables carry primary keys; cross-table joins on `line_id` and `candidate_part_id → part_id` execute and return data.
- The branch's `next_shift_forecast` migration/forecast ran and is queryable (8 plant rows) — developed on the Lakebase `dev` branch and promoted to `production`.
- `ops.work_orders` accepts INSERT/UPDATE/DELETE (and is the Lakebase CDF source — see `reverse_sync_sample.json`).
- `line_status`/`open_atrisk`/`maintenance_recommendations`/`parts` are managed synced mirrors (ONLINE via DLT), read-only by contract — distinct from the writable operational tables.